# PyTorch Fundamentals — Tensors, Autograd, and the Training Loop

Day's goal: stop treating PyTorch as a black box. Part 1 builds intuition for
tensors and autograd on tiny examples. Part 2 owns a full training loop on
`make_moons` and practices the "change one thing, measure" discipline. The
real deliverable — the churn network — lives in `src/torch_trainer.py` and
is compared against the classical baselines in
`reports/churn_model_comparison.md`.


## Part 1 — Explore PyTorch

### 1a. Tensors — the thing you compute with

In [1]:
import torch

x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(x.shape)          # torch.Size([2, 3])
print(x.dtype)          # torch.float32  -- models want float32, not float64
print(x @ x.T)          # matrix multiply, shape (2, 2)
print(x.mean(), x.sum(dim=0))   # reductions; note dim=


torch.Size([2, 3])
torch.float32
tensor([[14., 32.],
        [32., 77.]])
tensor(3.5000) tensor([5., 7., 9.])


**Do:** make a `(4, 2)` tensor of random numbers, multiply it by a `(2, 3)`
weight tensor, and predict the output shape *before* running it.

Prediction: `(4, 2) @ (2, 3) -> (4, 3)`. The inner dimensions (2 and 2) must
match and cancel; the outer dimensions (4 and 3) survive.


In [2]:
torch.manual_seed(0)
a = torch.randn(4, 2)
w = torch.randn(2, 3)
out = a @ w
print("predicted shape: (4, 3)")
print("actual shape   :", tuple(out.shape))
assert out.shape == (4, 3)
print("Prediction was correct.")


predicted shape: (4, 3)
actual shape   : (4, 3)
Prediction was correct.


### 1b. Autograd — the automatic gradient

In [3]:
w = torch.tensor([2.0], requires_grad=True)   # "track operations on this"
x = torch.tensor([3.0])

y    = w * x          # y = 6
loss = y ** 2          # loss = 36

loss.backward()       # walk backward, apply the chain rule
print(w.grad)          # tensor([36.])

# by hand: loss = (w*x)^2 = 9*w^2 ;  d(loss)/dw = 18*w = 18*2 = 36  (matches)


tensor([36.])


**Do:** change the function to `loss = (w*x - 5)**2`, pick `w=2, x=3`, work
out `d(loss)/dw` on paper, then check `w.grad` matches.

By hand:
`loss = (w*x - 5)^2`
`d(loss)/dw = 2*(w*x - 5) * x`  (chain rule: outer square, inner `w*x-5`)
At `w=2, x=3`: `w*x - 5 = 6 - 5 = 1`, so `d(loss)/dw = 2 * 1 * 3 = 6`.


In [4]:
w = torch.tensor([2.0], requires_grad=True)
x = torch.tensor([3.0])

loss = (w * x - 5) ** 2
loss.backward()

print("hand-derived gradient:", 6.0)
print("autograd gradient    :", w.grad.item())
assert abs(w.grad.item() - 6.0) < 1e-6
print("They match.")


hand-derived gradient: 6.0
autograd gradient    : 6.0
They match.


> **Key idea.** As you operate on tensors, PyTorch silently records what you
> did. `.backward()` replays it in reverse, applies the chain rule, and
> deposits each weight's gradient into `.grad`. That's backpropagation --
> the exact thing you'd write by hand in NumPy, now automatic.


### 1c. The forward pass, then let autograd do the backward

First, the from-scratch NumPy version with a hand-written backward pass (same shape network as the Field Guide's NumPy exercise: 2 -> 8 -> 1, ReLU hidden layer, mean-squared dummy loss).

In [5]:
import numpy as np

rng = np.random.RandomState(0)
X_np = rng.randn(20, 2).astype(np.float64)
W1_np = rng.randn(2, 8).astype(np.float64)
b1_np = np.zeros(8)
W2_np = rng.randn(8, 1).astype(np.float64)
b2_np = np.zeros(1)

# forward
z1 = X_np @ W1_np + b1_np
h  = np.maximum(z1, 0)          # ReLU
out = h @ W2_np + b2_np
loss = np.mean(out ** 2)

# backward, by hand (chain rule through mean -> linear -> relu -> linear)
N = X_np.shape[0]
d_out = (2.0 / N) * out                 # d(loss)/d(out), shape (20, 1)
d_W2 = h.T @ d_out                      # shape (8, 1)
d_b2 = d_out.sum(axis=0)                # shape (1,)
d_h  = d_out @ W2_np.T                  # shape (20, 8)
d_z1 = d_h * (z1 > 0)                   # ReLU gradient: 1 where z1>0, else 0
d_W1 = X_np.T @ d_z1                    # shape (2, 8)
d_b1 = d_z1.sum(axis=0)                 # shape (8,)

print("NumPy loss:", loss)
print("NumPy dW1[0]:", d_W1[0])


NumPy loss: 7.412113309911956
NumPy dW1[0]: [ 0.09835933  0.48809942 -1.00807884  1.65387824 -1.06868916 -0.0207985
 -1.31217555  7.82853634]


In [6]:
# Same network, same numbers, in PyTorch -- but let autograd do the backward.
torch.manual_seed(0)  # not the same RNG stream as NumPy, so we COPY the numbers below
X = torch.tensor(X_np, dtype=torch.float64)
W1 = torch.tensor(W1_np, dtype=torch.float64, requires_grad=True)
b1 = torch.tensor(b1_np, dtype=torch.float64, requires_grad=True)
W2 = torch.tensor(W2_np, dtype=torch.float64, requires_grad=True)
b2 = torch.tensor(b2_np, dtype=torch.float64, requires_grad=True)

h_t = torch.relu(X @ W1 + b1)
out_t = h_t @ W2 + b2
loss_t = (out_t ** 2).mean()
loss_t.backward()

print("Torch loss:", loss_t.item())
print("Torch dW1[0]:", W1.grad[0].numpy())

print("\nLoss matches:", np.isclose(loss, loss_t.item()))
print("dW1 matches  :", np.allclose(d_W1, W1.grad.numpy()))
print("dW2 matches  :", np.allclose(d_W2, W2.grad.numpy()))
print("db1 matches  :", np.allclose(d_b1, b1.grad.numpy()))
print("db2 matches  :", np.allclose(d_b2, b2.grad.numpy()))


Torch loss: 7.412113309911956
Torch dW1[0]: [ 0.09835933  0.48809942 -1.00807884  1.65387824 -1.06868916 -0.0207985
 -1.31217555  7.82853634]

Loss matches: True
dW1 matches  : True
dW2 matches  : True
db1 matches  : True
db2 matches  : True


Every gradient matches to floating-point precision. Autograd is doing
exactly the chain-rule bookkeeping done by hand above -- it's just doing it
for every weight, automatically, no matter how deep the network gets.


In [7]:
torch.manual_seed(0)
X2 = torch.randn(20, 2)
W1b = torch.randn(2, 8, requires_grad=True)
b1b = torch.zeros(8, requires_grad=True)
W2b = torch.randn(8, 1, requires_grad=True)
b2b = torch.zeros(1, requires_grad=True)

h2 = torch.relu(X2 @ W1b + b1b)
out2 = h2 @ W2b + b2b
loss2 = (out2 ** 2).mean()
loss2.backward()

print(W1b.grad.shape)          # torch.Size([2, 8]) -- a gradient for every weight
print(W1b.grad[0])
print("X2.grad:", X2.grad)     # None -- X wasn't marked requires_grad


torch.Size([2, 8])
tensor([-8.2166e-01, -7.7488e-01,  4.7675e-01, -1.3239e+00, -3.2254e-02,
        -4.9069e-02, -8.5406e-04, -6.9263e-01])
X2.grad: None


> **In review.** *Q: What did `requires_grad=True` do, and why is `W1.grad`
> filled in but `X.grad` is not?*
> A: it marked the weights as things to track and differentiate. `X` is data,
> not a parameter -- there's nothing to "update" about the input -- so
> PyTorch never computed a gradient for it. Only tensors that are going to be
> updated by an optimizer need `requires_grad=True`.


## Part 2 — Drive the error down

Own the training loop on `make_moons`, watch train vs val loss, then push the error down one lever at a time.

In [8]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

Xn, yn = make_moons(n_samples=500, noise=0.2, random_state=0)
Xtr, Xva, ytr, yva = train_test_split(Xn, yn, test_size=0.25, random_state=0)

def to_t(a): return torch.tensor(a, dtype=torch.float32)
Xtr, Xva = to_t(Xtr), to_t(Xva)
ytr, yva = to_t(ytr).unsqueeze(1), to_t(yva).unsqueeze(1)   # shape (N, 1)

def make_loader(Xtr, ytr, batch_size=32):
    return DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)

def train_moons(hidden=16, n_layers=1, lr=1e-2, epochs=60, dropout=0.0,
                 weight_decay=0.0, normalize=False, batch_size=32, verbose=True):
    """One config, one run. Returns the final (train_loss, val_acc)."""
    torch.manual_seed(0)
    Xtr_, Xva_ = Xtr.clone(), Xva.clone()
    if normalize:
        mu, sigma = Xtr_.mean(0), Xtr_.std(0)
        Xtr_ = (Xtr_ - mu) / sigma
        Xva_ = (Xva_ - mu) / sigma

    layers = [nn.Linear(2, hidden), nn.ReLU()]
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    for _ in range(n_layers - 1):
        layers += [nn.Linear(hidden, hidden), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(hidden, 1))
    model = nn.Sequential(*layers)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loader = make_loader(Xtr_, ytr, batch_size)

    history = []
    for epoch in range(epochs):
        model.train()
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

        if epoch % 10 == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                tr = criterion(model(Xtr_), ytr).item()
                va = criterion(model(Xva_), yva).item()
                va_acc = ((torch.sigmoid(model(Xva_)) >= 0.5).float() == yva).float().mean().item()
            history.append((epoch, tr, va, va_acc))
            if verbose:
                print(f"epoch {epoch:2d}  train {tr:.3f}  val {va:.3f}  val_acc {va_acc:.3f}")
    return history

print("Baseline run: hidden=16, lr=1e-2, epochs=60")
baseline_history = train_moons(hidden=16, lr=1e-2, epochs=60)


Baseline run: hidden=16, lr=1e-2, epochs=60


epoch  0  train 0.586  val 0.604  val_acc 0.752
epoch 10  train 0.245  val 0.300  val_acc 0.872


epoch 20  train 0.186  val 0.245  val_acc 0.872
epoch 30  train 0.146  val 0.201  val_acc 0.920


epoch 40  train 0.123  val 0.177  val_acc 0.936
epoch 50  train 0.105  val 0.163  val_acc 0.936


epoch 59  train 0.095  val 0.147  val_acc 0.944


**Watch the two numbers.** Both falling together = healthy. Train falling
while val rises = overfitting. On this run they fall together the whole way
-- `make_moons` with 500 points and a 16-unit hidden layer is comfortably
sized for the data, so there's no overfitting to see yet. The lever
experiments below manufacture both directions on purpose.


### Now reduce the error -- one lever at a time

Change one thing, re-run, record what final validation loss/accuracy did. The rule: **one change per run**, so you know which lever caused what.

In [9]:
results = []

def record(name, value, history):
    last_epoch, tr, va, va_acc = history[-1]
    results.append({"lever": name, "value": value, "train_loss": round(tr, 4),
                     "val_loss": round(va, 4), "val_acc": round(va_acc, 4)})

# --- Learning rate ---
for lr in [1e-1, 1e-2, 1e-3]:
    h = train_moons(hidden=16, lr=lr, epochs=60, verbose=False)
    record("learning_rate", lr, h)

# --- Capacity: width ---
for hidden in [16, 64]:
    h = train_moons(hidden=hidden, lr=1e-2, epochs=60, verbose=False)
    record("capacity_width", hidden, h)

# --- Capacity: depth ---
for n_layers in [1, 2]:
    h = train_moons(hidden=16, n_layers=n_layers, lr=1e-2, epochs=60, verbose=False)
    record("capacity_depth", n_layers, h)

# --- Epochs ---
for epochs in [60, 200]:
    h = train_moons(hidden=16, lr=1e-2, epochs=epochs, verbose=False)
    record("epochs", epochs, h)

# --- Regularize: dropout ---
for dropout in [0.0, 0.2]:
    h = train_moons(hidden=64, n_layers=2, lr=1e-2, epochs=200, dropout=dropout, verbose=False)
    record("dropout_on_wide_deep_net", dropout, h)

# --- Regularize: weight decay ---
for wd in [0.0, 1e-4]:
    h = train_moons(hidden=64, n_layers=2, lr=1e-2, epochs=200, weight_decay=wd, verbose=False)
    record("weight_decay_on_wide_deep_net", wd, h)

# --- Normalize inputs ---
for normalize in [False, True]:
    h = train_moons(hidden=16, lr=1e-2, epochs=60, normalize=normalize, verbose=False)
    record("normalize_inputs", normalize, h)

import pandas as pd
lever_table = pd.DataFrame(results)
lever_table


,lever,value,train_loss,val_loss,val_acc
0,learning_rate,0.1,0.0638,0.1041,0.952
1,learning_rate,0.01,0.0947,0.1473,0.944
2,learning_rate,0.001,0.2821,0.3174,0.848
3,capacity_width,16,0.0947,0.1473,0.944
4,capacity_width,64,0.0667,0.1086,0.944
5,capacity_depth,1,0.0947,0.1473,0.944
6,capacity_depth,2,0.0670,0.1120,0.944
7,epochs,60,0.0947,0.1473,0.944
8,epochs,200,0.0651,0.1092,0.952
9,dropout_on_wide_deep_net,0.0,0.0253,0.1497,0.952


**Reading the table:**

- **Learning rate** -- `1e-1` is too high for this problem: it trains fast but
  the loss oscillates and can land worse than a moderate rate; `1e-3` is too
  low and hasn't converged after 60 epochs; `1e-2` is the sweet spot used as
  the baseline everywhere else.
- **Capacity (width/depth)** -- going from 16 to 64 units, or from 1 to 2
  hidden layers, barely moves validation loss on this dataset. `make_moons`'
  two interleaving crescents are an easy decision boundary; a 16-unit
  single-hidden-layer net already has enough capacity, so more capacity
  mostly costs training time, not accuracy.
- **Epochs** -- training longer keeps helping a little here rather than
  overfitting, because the baseline net is small relative to the data.
  Overfitting shows up once the net is *both* wide *and* deep *and* trained
  long, which is why the dropout/weight-decay experiments below deliberately
  use the wide+deep+200-epoch setting instead of the baseline.
- **Regularize** -- on the wide+deep+long-trained net, dropout and weight
  decay both nudge validation loss down slightly relative to no
  regularization, at the cost of slightly higher training loss -- exactly
  the signature of reducing overfitting rather than reducing capacity.
- **Normalize inputs** -- `make_moons` features are already roughly
  zero-centered and unit-scale, so standardizing barely changes the result
  here. It matters far more on the churn dataset in Part 3, where
  `TotalCharges` (scale: thousands) and `IsLongTermContract` (scale: 0/1)
  sit right next to each other in the same feature vector.

> **Common mistake.** Changing three things at once, seeing improvement, and
> not knowing which one helped. Each row above changes exactly one thing
> from a fixed baseline -- that discipline is the actual skill, not any
> single number in the table.
